# 基于 MindSpore 的 TorchDrug 论文简化复现

我选的论文是 **TorchDrug: A Powerful and Flexible Machine Learning Platform for Drug Discovery**。这个 Notebook 主要记录我把论文里的分子性质预测流程简化后，用 MindSpore 重新实现 GIN/GAT 的过程。这里没有直接调用 PyTorch 或 TorchDrug 的训练代码，只借鉴了 TorchDrug 的图表示和特征设计。

## 1. 检查项目路径

先把项目根目录加入 `sys.path`，这样后面的代码可以直接导入 `src` 里的数据处理、模型和训练函数。

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("Python:", sys.version)

## 2. 导入依赖和本项目代码

这一部分主要导入 MindSpore、NumPy 和我自己写的几个模块。`dataset.py` 负责数据读取和分子图构造，`models.py` 里是 GIN/GAT，`trainer.py` 里放训练和评估循环。

In [ ]:
import mindspore as ms
import numpy as np

from src.dataset import load_dataset, split_dataset
from src.models import build_model
from src.trainer import TrainConfig, fit

DEVICE_TARGET = "GPU"   # 华为云 CUDA 镜像用 GPU；如果只用 CPU，把这里改成 "CPU"

ms.set_context(mode=ms.PYNATIVE_MODE, device_target=DEVICE_TARGET)
ms.set_seed(0)
np.random.seed(0)

## 3. 读取数据并生成分子图

这里先用 BACE 做演示。代码会下载公开 CSV，然后用 RDKit 把 SMILES 转成分子图。节点特征和边特征参考了 TorchDrug 的默认设置。要跑 HIV 时，把 `dataset_name` 改成 `"hiv"` 即可。

In [ ]:
dataset_name = "bace"
dataset = load_dataset(dataset_name, PROJECT_ROOT / "data")

print("Dataset:", dataset.name)
print("Num molecules:", len(dataset))
print("Node feature dim:", dataset.node_feature_dim)
print("Edge feature dim:", dataset.edge_feature_dim)
print("First SMILES:", dataset[0]["smiles"])
print("First graph nodes:", dataset[0]["node_feature"].shape[0])

## 4. 划分训练集、验证集和测试集

BACE 这里使用 scaffold split，比例是 8:1:1。这样测试集和训练集的分子骨架差异更大，比普通随机划分更能看出模型对新结构的泛化情况。

In [ ]:
train_set, valid_set, test_set = split_dataset(dataset, split="scaffold", seed=0)
print("Train / Valid / Test:", len(train_set), len(valid_set), len(test_set))

## 5. 构建 GIN 模型

GIN 的核心是对邻居信息做求和聚合，再用 MLP 更新节点表示。这里的 `torchdrug_like` 版本加入了边特征、BatchNorm、shortcut、concat hidden 和 sum readout，让结构尽量贴近 TorchDrug 中的写法。

In [ ]:
gin_model = build_model(
    "gin",
    input_dim=dataset.node_feature_dim,
    edge_input_dim=dataset.edge_feature_dim,
    hidden_dim=128,
    num_layer=4,
    num_head=4,
    dropout=0.1,
    variant="torchdrug_like",
)
print(gin_model)

## 6. 训练并评估 GIN

为了让前面的流程检查快一点，这里只训练 3 个 epoch。第 8 节会在 Notebook 里跑四组完整实验，并把结果保存成 CSV。

In [ ]:
config = TrainConfig(epoch=3, batch_size=32, lr=1e-3, seed=0)
valid_metrics, test_metrics, selected_epoch = fit(gin_model, train_set, valid_set, test_set, config)

print("Selected epoch:", selected_epoch)
print("Best valid:", valid_metrics)
print("Matched test:", test_metrics)

## 7. 构建并训练 GAT 模型

GAT 会给不同邻居分配不同注意力权重。这里同样保留边特征输入，把化学键信息加到 attention 计算里。

In [ ]:
gat_model = build_model(
    "gat",
    input_dim=dataset.node_feature_dim,
    edge_input_dim=dataset.edge_feature_dim,
    hidden_dim=128,
    num_layer=4,
    num_head=4,
    dropout=0.1,
    variant="torchdrug_like",
)

config = TrainConfig(epoch=3, batch_size=32, lr=1e-3, seed=0)
valid_metrics, test_metrics, selected_epoch = fit(gat_model, train_set, valid_set, test_set, config)

print("Selected epoch:", selected_epoch)
print("Best valid:", valid_metrics)
print("Matched test:", test_metrics)

## 8. 在 Notebook 中跑四组完整实验

这一节直接在 Notebook 里跑 BACE/HIV 和 GIN/GAT 的四组实验。这样提交时老师打开 Notebook 就能看到完整训练流程；`run_experiment.py` 仍然保留，作为命令行复现实验的入口。

In [ ]:
from datetime import datetime
import random
import pandas as pd

FULL_EPOCH = 100
SEED = 0
LR = 1e-3
HIDDEN_DIM = 256
NUM_LAYER = 4
NUM_HEAD = 4
DROPOUT = 0.1
RESULT_PATH = PROJECT_ROOT / "results" / "notebook_experiment_results.csv"

experiment_plan = [
    {"dataset": "bace", "model": "gin", "split": "scaffold", "batch_size": 256},
    {"dataset": "bace", "model": "gat", "split": "scaffold", "batch_size": 256},
    {"dataset": "hiv", "model": "gin", "split": "random", "batch_size": 512},
    {"dataset": "hiv", "model": "gat", "split": "random", "batch_size": 512},
]

def reset_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    ms.set_seed(seed)

def run_one_notebook_experiment(item):
    reset_seed(SEED)
    ms.set_context(mode=ms.PYNATIVE_MODE, device_target=DEVICE_TARGET)

    dataset = load_dataset(item["dataset"], PROJECT_ROOT / "data", feature_set="torchdrug_default")
    train_set, valid_set, test_set = split_dataset(dataset, split=item["split"], seed=SEED)

    model = build_model(
        item["model"],
        input_dim=dataset.node_feature_dim,
        edge_input_dim=dataset.edge_feature_dim,
        hidden_dim=HIDDEN_DIM,
        num_layer=NUM_LAYER,
        num_head=NUM_HEAD,
        dropout=DROPOUT,
        variant="torchdrug_like",
        readout="sum",
        num_mlp_layer=1,
    )
    config = TrainConfig(
        epoch=FULL_EPOCH,
        batch_size=item["batch_size"],
        lr=LR,
        seed=SEED,
        selection="best_valid_auroc",
    )
    valid_metrics, test_metrics, selected_epoch = fit(model, train_set, valid_set, test_set, config)

    return {
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "framework": "MindSpore",
        "dataset": item["dataset"],
        "model": item["model"],
        "variant": "torchdrug_like",
        "graph_format": "edge_list",
        "feature_set": "torchdrug_default",
        "seed": SEED,
        "split": item["split"],
        "selection": "best_valid_auroc",
        "selected_epoch": selected_epoch,
        "epoch": FULL_EPOCH,
        "batch_size": item["batch_size"],
        "hidden_dim": HIDDEN_DIM,
        "num_layer": NUM_LAYER,
        "num_head": NUM_HEAD,
        "readout": "sum",
        "num_mlp_layer": 1,
        "node_feature_dim": dataset.node_feature_dim,
        "edge_feature_dim": dataset.edge_feature_dim,
        "valid_auroc": valid_metrics["auroc"],
        "valid_auprc": valid_metrics["auprc"],
        "test_auroc": test_metrics["auroc"],
        "test_auprc": test_metrics["auprc"],
    }

notebook_results = []
for item in experiment_plan:
    print(f"\n=== dataset={item['dataset']} model={item['model']} split={item['split']} ===")
    notebook_results.append(run_one_notebook_experiment(item))

result_df = pd.DataFrame(notebook_results)
RESULT_PATH.parent.mkdir(parents=True, exist_ok=True)
result_df.to_csv(RESULT_PATH, index=False)
print("Saved results to:", RESULT_PATH)
result_df


## 9. 结果记录

第 8 节会把 Notebook 运行得到的结果保存到 `results/notebook_experiment_results.csv`。如果用终端脚本跑，结果会保存到 `results/experiment_results.csv`。两个文件的字段基本一致，报告里选其中一份完整结果整理即可。